In [17]:
# ============================================================
# FINAL MULTI-CANCER MULTI-OMICS PIPELINE
# Classification + Explainability + Visualization
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score
from sklearn.utils.class_weight import compute_class_weight
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================

SEED = 42
MAX_EPOCHS = 400
MIN_EPOCHS = 100
PATIENCE = 45
LR = 1e-3
WEIGHT_DECAY = 5e-4

ALIGN_W = 0.2
GATE_ENT_W = 0.05
OMICS_DROPOUT_P = 0.2

TOP_K_GENES = 20
N_SPLITS = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

BASE_DIR = "./preprocessing/processed_multicancer"
RESULTS_ROOT = "./results_all_cancers"
os.makedirs(RESULTS_ROOT, exist_ok=True)

GS_TYPES = [
    "GS-BRCA",
    "GS-LGG",
    "GS-OV",
    "GS-COAD",
    "GS-GBM"
]

OMICS = ["mRNA", "miRNA", "CNV", "Methy"]

# ============================================================
# LOSSES
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n = logits.size(1)
        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smoothing / (n - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)

        log_p = F.log_softmax(logits, dim=1)
        p = torch.exp(log_p)
        focal = (1 - p) ** self.gamma
        loss = -true_dist * focal * log_p

        if self.alpha is not None:
            loss = loss * self.alpha

        return loss.sum(dim=1).mean()

def alignment_loss(zs):
    loss = 0
    for i in range(len(zs)):
        for j in range(i + 1, len(zs)):
            zi = F.normalize(zs[i], dim=1)
            zj = F.normalize(zs[j], dim=1)
            loss += 1 - (zi * zj).sum(dim=1).mean()
    return loss

def gate_entropy(g):
    eps = 1e-6
    return -(g * torch.log(g + eps) + (1 - g) * torch.log(1 - g + eps)).mean()

# ============================================================
# MODEL
# ============================================================

class OmicsEncoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class GateNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.fc(z)

class GatedMultiOmicsClassifier(nn.Module):
    def __init__(self, in_dims, num_classes):
        super().__init__()
        self.encoders = nn.ModuleDict({k: OmicsEncoder(v) for k, v in in_dims.items()})
        self.gates = nn.ModuleDict({k: GateNet() for k in in_dims})

        self.classifier = nn.Sequential(
            nn.Linear(128 * len(in_dims), 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_dict, training=False):
        zs, gated, gates = [], [], {}

        for m in self.encoders:
            x = x_dict[m]
            if training and torch.rand(1).item() < OMICS_DROPOUT_P:
                x = torch.zeros_like(x)

            z = self.encoders[m](x)
            g = self.gates[m](z)

            zs.append(z)
            gated.append(z * g)
            gates[m] = g

        fused = torch.cat(gated, dim=1)
        logits = self.classifier(fused)
        return logits, zs, gates

# ============================================================
# TRAIN ONE SPLIT
# ============================================================

def train_on_split(omics_data, y, tr, te):
    Xtr, Xte = {}, {}
    for m, X in omics_data.items():
        sc = StandardScaler()
        Xtr[m] = sc.fit_transform(X[tr])
        Xte[m] = sc.transform(X[te])

    ytr, yte = y[tr], y[te]

    Xtr_t = {m: torch.tensor(v, dtype=torch.float32).to(DEVICE) for m, v in Xtr.items()}
    Xte_t = {m: torch.tensor(v, dtype=torch.float32).to(DEVICE) for m, v in Xte.items()}
    ytr_t = torch.tensor(ytr, dtype=torch.long).to(DEVICE)

    w = compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)
    w = torch.tensor(w, dtype=torch.float32).to(DEVICE)

    loss_fn = FocalLoss(alpha=w)

    in_dims = {m: Xtr[m].shape[1] for m in Xtr}
    model = GatedMultiOmicsClassifier(in_dims, len(np.unique(y))).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_prec, wait = 0, 0
    best_state = None

    for epoch in range(MAX_EPOCHS):
        model.train()
        opt.zero_grad()

        logits, zs, gates = model(Xtr_t, training=True)
        loss = (
            loss_fn(logits, ytr_t)
            + ALIGN_W * alignment_loss(zs)
            + GATE_ENT_W * sum(gate_entropy(g) for g in gates.values())
        )

        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            preds = model(Xte_t)[0].argmax(1).cpu().numpy()
            prec = precision_score(yte, preds, average="weighted", zero_division=0)

        if prec > best_prec:
            best_prec = prec
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch >= MIN_EPOCHS and wait >= PATIENCE:
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    return best_prec, model, Xte_t

# ============================================================
# MULTI-CANCER LOOP
# ============================================================

for CANCER in GS_TYPES:
    print(f"\n==================== {CANCER} ====================")

    DATA_DIR = os.path.join(BASE_DIR, CANCER)
    OUT_DIR = os.path.join(RESULTS_ROOT, CANCER)
    os.makedirs(OUT_DIR, exist_ok=True)

    try:
        omics_data = {
            "mRNA":  np.load(os.path.join(DATA_DIR, "mRNA_processed.npy")),
            "miRNA": np.load(os.path.join(DATA_DIR, "miRNA_processed.npy")),
            "CNV":   np.load(os.path.join(DATA_DIR, "CNV_processed.npy")),
            "Methy": np.load(os.path.join(DATA_DIR, "Methy_processed.npy")),
        }
        y = np.load(os.path.join(DATA_DIR, "labels.npy"))
    except FileNotFoundError:
        print(f"Skipping {CANCER} (missing files)")
        continue

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_scores = []
    gate_means = defaultdict(list)
    latent_collect = defaultdict(list)

    for tr, te in skf.split(omics_data["mRNA"], y):
        prec, model, Xte_t = train_on_split(omics_data, y, tr, te)
        fold_scores.append(prec)

        model.eval()
        with torch.no_grad():
            _, zs, gates = model(Xte_t)
            for i, m in enumerate(OMICS):
                gate_means[m].append(gates[m].mean().item())
                latent_collect[m].append(torch.abs(zs[i]).mean(dim=0).cpu().numpy())

    # ========================================================
    # SAVE PERFORMANCE
    # ========================================================

    perf_df = pd.DataFrame({
        "Mean_Precision": [np.mean(fold_scores)],
        "Std_Precision": [np.std(fold_scores)]
    })
    perf_df.to_csv(os.path.join(OUT_DIR, "performance.csv"), index=False)

    # ========================================================
    # GATE IMPORTANCE
    # ========================================================

    mean_gate = {m: np.mean(gate_means[m]) for m in OMICS}
    pd.DataFrame(mean_gate.items(), columns=["Omics", "Mean_Gate"]).to_csv(
        os.path.join(OUT_DIR, "gate_importance.csv"), index=False
    )

    plt.figure(figsize=(6,4))
    plt.bar(mean_gate.keys(), mean_gate.values())
    plt.title(f"{CANCER}: Gate Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "gate_importance.png"), dpi=300)
    plt.close()

    # ========================================================
    # FAEC
    # ========================================================

    faec = {m: np.std(gate_means[m]) for m in OMICS}
    plt.figure(figsize=(6,4))
    plt.bar(faec.keys(), faec.values())
    plt.title(f"{CANCER}: FAEC (Gate Stability)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "faec_gate_stability.png"), dpi=300)
    plt.close()

    # ========================================================
    # CORI
    # ========================================================

    mean_latent = {m: np.mean(latent_collect[m], axis=0) for m in OMICS}
    cori_rows = []

    for i in range(len(OMICS)):
        for j in range(i + 1, len(OMICS)):
            v1 = np.sort(mean_latent[OMICS[i]])[::-1]
            v2 = np.sort(mean_latent[OMICS[j]])[::-1]
            L = min(len(v1), len(v2))
            corr = np.corrcoef(v1[:L], v2[:L])[0,1]
            cori_rows.append({
                "Pair": f"{OMICS[i]}-{OMICS[j]}",
                "CORI": abs(corr)
            })

    cori_df = pd.DataFrame(cori_rows)
    cori_df.to_csv(os.path.join(OUT_DIR, "cori.csv"), index=False)

    plt.figure(figsize=(7,4))
    plt.bar(cori_df["Pair"], cori_df["CORI"])
    plt.xticks(rotation=30)
    plt.title(f"{CANCER}: CORI")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "cori_plot.png"), dpi=300)
    plt.close()

    # ========================================================
    # TOP-20 GENES (Sensitivity)
    # ========================================================

    for m in OMICS:
        encoder = model.encoders[m]
        X = Xte_t[m].clone().requires_grad_(True)
        z = encoder(X)
        score = torch.norm(z, dim=1).mean()
        score.backward()

        sens = X.grad.abs().mean(dim=0).cpu().numpy()
        idx = np.argsort(sens)[::-1][:TOP_K_GENES]

        pd.DataFrame({
            "Feature_Index": idx,
            "Sensitivity": sens[idx]
        }).to_csv(
            os.path.join(OUT_DIR, f"top_{TOP_K_GENES}_{m}_genes.csv"),
            index=False
        )

    print(f"{CANCER} DONE")

print("\nALL CANCERS PROCESSED SUCCESSFULLY")



==================== GS-BRCA ====================
GS-BRCA DONE

==================== GS-LGG ====================
GS-LGG DONE

==================== GS-OV ====================
GS-OV DONE

==================== GS-COAD ====================


/opt/miniconda3/envs/mlomics/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


GS-COAD DONE

==================== GS-GBM ====================
GS-GBM DONE

ALL CANCERS PROCESSED SUCCESSFULLY
